# Prototipo del filtro temporal de movimiento

Este notebook experimenta con el Bloque B de la issue #9 antes de trasladar la lógica definitiva a `src/evaluation/motion_filter.py`. Usa únicamente datos sintéticos pequeños para que Saúl y Dolly puedan comprobar manualmente cada decisión.

**Objetivo:** seguir vehículos entre frames compensando el movimiento de cámara y eliminar únicamente tracks observados durante al menos 10 frames cuyo desplazamiento neto compensado sea menor de 8 px.

**Estado:** prototipo experimental completo; todavía no sustituye al módulo de producción.

## 1. Contrato y umbrales

La entrada contiene detecciones OBB agrupadas por frame y una homografía para cada transición. La matriz asociada al frame actual proyecta coordenadas del frame anterior hacia el actual.

Reglas obligatorias: asociación estrictamente menor de 30 px; track estático solo si dura al menos 10 frames y su desplazamiento compensado es estrictamente menor de 8 px. `static_vehicles.json` no es una entrada.

In [ ]:
import math
import warnings

import cv2
import numpy as np

MAX_ASSOCIATION_DISTANCE_PX = 30.0
MIN_STATIC_TRACK_FRAMES = 10
MAX_STATIC_DISPLACEMENT_PX = 8.0

## 2. Secuencia sintética conocida

Creamos 12 frames y una cámara que desplaza toda la imagen 2 px hacia la derecha en cada transición. El vehículo `static` solo reproduce ese desplazamiento de cámara; `moving` añade 3 px propios por frame; `short` es estático pero aparece únicamente 6 frames, por lo que no debe eliminarse.

In [ ]:
def experimental_detection(name, class_id, cx, cy, score=0.90):
    return {
        "name": name,
        "class_id": class_id,
        "score": score,
        "obb": (float(cx), float(cy), 40.0, 20.0, 0.0),
    }


frame_ids = [f"clip_demo_{index:04d}" for index in range(12)]
camera_step = np.array(
    [[1.0, 0.0, 2.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]],
    dtype=np.float64,
)
homographies_by_frame = {
    frame_id: camera_step.copy() for frame_id in frame_ids[1:]
}
predictions_by_frame = {}
for frame_index, frame_id in enumerate(frame_ids):
    detections = [
        experimental_detection("static", 1, 100.0 + 2.0 * frame_index, 120.0),
        experimental_detection("moving", 2, 250.0 + 5.0 * frame_index, 180.0),
    ]
    if frame_index < 6:
        detections.append(
            experimental_detection("short", 3, 380.0 + 2.0 * frame_index, 240.0)
        )
    predictions_by_frame[frame_id] = detections

assert len(predictions_by_frame) == 12
assert sum(map(len, predictions_by_frame.values())) == 30
print("Frames: 12 | detecciones: 30 | homografías: 11")

## 3. Proyección de centroides por homografía

Un centro `(x, y)` se convierte a `(x, y, 1)`, se multiplica por la matriz `3×3` y se divide entre la coordenada homogénea `w`. Una homografía inválida usa identidad con advertencia para no romper todo el clip.

In [ ]:
def experimental_valid_homography(homography):
    if homography is None:
        warnings.warn("Homografía ausente; se usará identidad", stacklevel=2)
        return np.eye(3, dtype=np.float64)
    matrix = np.asarray(homography, dtype=np.float64)
    if (
        matrix.shape != (3, 3)
        or not np.all(np.isfinite(matrix))
        or abs(np.linalg.det(matrix)) < 1e-12
    ):
        warnings.warn("Homografía inválida; se usará identidad", stacklevel=2)
        return np.eye(3, dtype=np.float64)
    return matrix


def experimental_project_centroid(centroid, homography):
    matrix = experimental_valid_homography(homography)
    point = np.array([centroid[0], centroid[1], 1.0], dtype=np.float64)
    projected = matrix @ point
    if abs(projected[2]) < 1e-12 or not np.all(np.isfinite(projected)):
        warnings.warn("Proyección inválida; se conservará el centro", stacklevel=2)
        return (float(centroid[0]), float(centroid[1]))
    return (float(projected[0] / projected[2]), float(projected[1] / projected[2]))


projected_static = experimental_project_centroid((100.0, 120.0), camera_step)
assert np.allclose(projected_static, (102.0, 120.0))
assert np.allclose(experimental_project_centroid((5.0, 7.0), None), (5.0, 7.0))
print("Centro anterior (100, 120) → centro proyectado", projected_static)

## 4. Asociación greedy uno a uno

Se calculan candidatos únicamente entre la misma clase. Se ordenan por distancia y se aceptan greedily sin reutilizar track ni detección. La comparación es estricta: `29.9 < 30` se acepta y `30.0 < 30` se rechaza.

In [ ]:
def experimental_associate_detections(
    active_tracks, detections, homography, max_distance_px=30.0
):
    candidates = []
    for track_id in sorted(active_tracks):
        track = active_tracks[track_id]
        projected = experimental_project_centroid(track["last_center"], homography)
        for detection_index, detection in enumerate(detections):
            if detection["class_id"] != track["class_id"]:
                continue
            cx, cy = detection["obb"][:2]
            distance = math.hypot(cx - projected[0], cy - projected[1])
            candidates.append((distance, track_id, detection_index, projected))

    matches = []
    used_tracks = set()
    used_detections = set()
    for distance, track_id, detection_index, projected in sorted(candidates):
        if distance >= max_distance_px:
            continue
        if track_id in used_tracks or detection_index in used_detections:
            continue
        matches.append(
            {
                "track_id": track_id,
                "detection_index": detection_index,
                "distance_px": distance,
                "projected_center": projected,
            }
        )
        used_tracks.add(track_id)
        used_detections.add(detection_index)
    return matches


boundary_track = {0: {"class_id": 1, "last_center": (0.0, 0.0)}}
accepted = experimental_associate_detections(
    boundary_track, [experimental_detection("near", 1, 29.9, 0.0)], np.eye(3)
)
rejected = experimental_associate_detections(
    boundary_track, [experimental_detection("limit", 1, 30.0, 0.0)], np.eye(3)
)
assert len(accepted) == 1
assert rejected == []
print("29.9 px: asociado | 30.0 px: rechazado")

## 5. Construcción secuencial de tracks

El tracker se actualiza dentro del bucle de frames. Un track solo permanece activo si obtuvo una detección en el frame actual; las detecciones libres crean nuevos tracks. Cada observación guarda `frame_id` e índice para poder eliminar exactamente la predicción original.

In [ ]:
def experimental_build_tracks(
    predictions_by_frame, homographies_by_frame, max_distance_px=30.0
):
    tracks = {}
    active_track_ids = set()
    next_track_id = 0

    for frame_position, (frame_id, detections) in enumerate(predictions_by_frame.items()):
        active_tracks = {track_id: tracks[track_id] for track_id in active_track_ids}
        homography = None if frame_position == 0 else homographies_by_frame.get(frame_id)
        matches = (
            []
            if frame_position == 0
            else experimental_associate_detections(
                active_tracks, detections, homography, max_distance_px
            )
        )
        matched_detection_indices = set()
        current_track_ids = set()

        for match in matches:
            track_id = match["track_id"]
            detection_index = match["detection_index"]
            detection = detections[detection_index]
            cx, cy = detection["obb"][:2]
            tracks[track_id]["last_center"] = (cx, cy)
            tracks[track_id]["observations"].append(
                {
                    "frame_id": frame_id,
                    "detection_index": detection_index,
                    "detection": detection,
                }
            )
            matched_detection_indices.add(detection_index)
            current_track_ids.add(track_id)

        for detection_index, detection in enumerate(detections):
            if detection_index in matched_detection_indices:
                continue
            cx, cy = detection["obb"][:2]
            tracks[next_track_id] = {
                "track_id": next_track_id,
                "class_id": detection["class_id"],
                "last_center": (cx, cy),
                "observations": [
                    {
                        "frame_id": frame_id,
                        "detection_index": detection_index,
                        "detection": detection,
                    }
                ],
            }
            current_track_ids.add(next_track_id)
            next_track_id += 1

        active_track_ids = current_track_ids
    return tracks


experimental_tracks = experimental_build_tracks(
    predictions_by_frame, homographies_by_frame
)
assert len(experimental_tracks) == 3
for track_id, track in experimental_tracks.items():
    name = track["observations"][0]["detection"]["name"]
    print(f"track={track_id} nombre={name} frames={len(track['observations'])}")

## 6. Desplazamiento compensado

Para cada transición proyectamos el centro anterior mediante la homografía. El vector residual entre esa proyección y la detección actual representa movimiento propio del vehículo. Sumamos los residuos y calculamos su norma: estático = 0 px; móvil = 33 px en once transiciones.

In [ ]:
def experimental_compensated_displacement(track, homographies_by_frame):
    residual_x = 0.0
    residual_y = 0.0
    observations = track["observations"]
    for previous, current in zip(observations, observations[1:]):
        previous_center = previous["detection"]["obb"][:2]
        current_center = current["detection"]["obb"][:2]
        projected = experimental_project_centroid(
            previous_center, homographies_by_frame.get(current["frame_id"])
        )
        residual_x += current_center[0] - projected[0]
        residual_y += current_center[1] - projected[1]
    return math.hypot(residual_x, residual_y)


track_by_name = {
    track["observations"][0]["detection"]["name"]: track
    for track in experimental_tracks.values()
}
displacements = {
    name: experimental_compensated_displacement(track, homographies_by_frame)
    for name, track in track_by_name.items()
}
assert math.isclose(displacements["static"], 0.0)
assert math.isclose(displacements["moving"], 33.0)
assert math.isclose(displacements["short"], 0.0)
print(displacements)

## 7. Clasificación y filtrado

Un track se elimina solamente si cumple simultáneamente duración `>= 10` y desplazamiento `< 8`. El track corto no se elimina aunque parezca inmóvil, porque no aporta evidencia temporal suficiente.

In [ ]:
def experimental_classify_static_tracks(
    tracks, homographies_by_frame, min_frames=10, max_displacement_px=8.0
):
    static_track_ids = set()
    for track_id, track in tracks.items():
        duration = len(track["observations"])
        displacement = experimental_compensated_displacement(
            track, homographies_by_frame
        )
        if duration >= min_frames and displacement < max_displacement_px:
            static_track_ids.add(track_id)
    return static_track_ids


def experimental_filter_static_predictions(
    predictions_by_frame, homographies_by_frame
):
    tracks = experimental_build_tracks(predictions_by_frame, homographies_by_frame)
    static_track_ids = experimental_classify_static_tracks(
        tracks, homographies_by_frame
    )
    removed_references = {
        (observation["frame_id"], observation["detection_index"])
        for track_id in static_track_ids
        for observation in tracks[track_id]["observations"]
    }
    filtered = {
        frame_id: [
            detection
            for detection_index, detection in enumerate(detections)
            if (frame_id, detection_index) not in removed_references
        ]
        for frame_id, detections in predictions_by_frame.items()
    }
    diagnostics = {
        "total_tracks": len(tracks),
        "static_track_ids": tuple(sorted(static_track_ids)),
        "retained_track_ids": tuple(sorted(set(tracks) - static_track_ids)),
        "removed_predictions": len(removed_references),
    }
    return filtered, diagnostics


filtered_predictions, filter_diagnostics = experimental_filter_static_predictions(
    predictions_by_frame, homographies_by_frame
)
remaining_names = {
    detection["name"]
    for detections in filtered_predictions.values()
    for detection in detections
}
assert filter_diagnostics["static_track_ids"] == (0,)
assert filter_diagnostics["removed_predictions"] == 12
assert remaining_names == {"moving", "short"}
print(filter_diagnostics)

## 8. Visualización de trayectorias

Las líneas muestran las posiciones observadas en la imagen. Todas avanzan por el movimiento de cámara, pero la compensación revela que solo `moving` acumula movimiento propio.

In [ ]:
trajectory_canvas = np.full((320, 520, 3), 245, dtype=np.uint8)
colors = {"static": (50, 180, 50), "moving": (40, 70, 220), "short": (220, 140, 40)}
for name, track in track_by_name.items():
    points = [
        tuple(map(lambda value: int(round(value)), obs["detection"]["obb"][:2]))
        for obs in track["observations"]
    ]
    for start, end in zip(points, points[1:]):
        cv2.line(trajectory_canvas, start, end, colors[name], 2)
    for point in points:
        cv2.circle(trajectory_canvas, point, 4, colors[name], -1)
    cv2.putText(
        trajectory_canvas, name, (points[0][0] - 20, points[0][1] - 12),
        cv2.FONT_HERSHEY_SIMPLEX, 0.55, colors[name], 2, cv2.LINE_AA
    )

try:
    from IPython.display import Image, display

    encoded_ok, encoded_canvas = cv2.imencode(".png", trajectory_canvas)
    assert encoded_ok
    display(Image(data=encoded_canvas.tobytes()))
except ImportError:
    print("Visualización creada:", trajectory_canvas.shape)

## 9. Visualización de la compensación

En la primera transición, el centro verde es la posición anterior, el círculo amarillo es su proyección por cámara y la cruz roja es la detección actual. Para el vehículo estático, proyección y detección coinciden; para el móvil queda un residuo de 3 px.

In [ ]:
projection_canvas = np.full((280, 440, 3), 245, dtype=np.uint8)
for name, previous_center, current_center in (
    ("static", (100.0, 120.0), (102.0, 120.0)),
    ("moving", (250.0, 180.0), (255.0, 180.0)),
):
    projected_center = experimental_project_centroid(previous_center, camera_step)
    previous_point = tuple(map(int, previous_center))
    projected_point = tuple(map(int, projected_center))
    current_point = tuple(map(int, current_center))
    cv2.circle(projection_canvas, previous_point, 7, (40, 170, 40), -1)
    cv2.circle(projection_canvas, projected_point, 10, (0, 190, 220), 2)
    cv2.drawMarker(
        projection_canvas, current_point, (30, 30, 220),
        cv2.MARKER_TILTED_CROSS, 14, 2
    )
    cv2.putText(
        projection_canvas, name, (previous_point[0] - 30, previous_point[1] - 18),
        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (30, 30, 30), 1, cv2.LINE_AA
    )

try:
    from IPython.display import Image, display

    encoded_ok, encoded_projection = cv2.imencode(".png", projection_canvas)
    assert encoded_ok
    display(Image(data=encoded_projection.tobytes()))
except ImportError:
    print("Visualización creada:", projection_canvas.shape)

## 10. Decisiones que se migrarán a `motion_filter.py`

1. Procesar frames secuencialmente y actualizar el tracker dentro del bucle.
2. Asociar solamente detecciones de la misma clase y usar matching greedy uno a uno con distancia `< 30 px`.
3. Compensar la cámara proyectando el centro anterior con la homografía anterior→actual.
4. Usar identidad y advertencia si una homografía falta o es inválida.
5. Eliminar únicamente tracks con duración `>= 10` y desplazamiento compensado `< 8 px`.
6. Filtrar mediante referencias `(frame_id, detection_index)`, no comparando floats de OBB.
7. Devolver diagnósticos auditables y no depender de `static_vehicles.json`.
8. Cuando se estime homografía con ORB en producción, llamar `detectAndCompute(image, mask)` con la máscara como argumento posicional para conservar compatibilidad con OpenCV de Colab.

Resultados esperados del prototipo: `static` se elimina (12 frames, 0 px); `moving` se conserva (12 frames, 33 px); `short` se conserva (6 frames, 0 px).